# Uber Fare Prediction

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:

df = pd.read_csv("uber.csv")

print(df.head())
print(df.shape)


In [ ]:

print(df.head())

In [ ]:
# Remove unnecessary columns
df = df.drop(columns=['Unnamed: 0', 'key'], errors='ignore')

# Convert date into hour
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], utc=True)
df['hour'] = df['pickup_datetime'].dt.hour

# Remove date column
df = df.drop('pickup_datetime', axis=1)

# Remove missing and duplicate values
df = df.dropna().drop_duplicates()

# Remove invalid values
df = df[(df.fare_amount > 0) & (df.passenger_count.between(1, 6))]

print(df.head())
print(df.shape)

In [ ]:
# Calculate IQR
Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

# Remove outliers
df = df[df['fare_amount'].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)]

# Display boxplot
sns.boxplot(x=df['fare_amount'])
plt.show()

In [ ]:
# Show correlation between columns
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
# Separate input and output
X = df.drop('fare_amount', axis=1)
y = df['fare_amount']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Create and train model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict fares
p1 = lr.predict(X_test)

In [ ]:
# Create and train model
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)

# Predict fares
p2 = rf.predict(X_test)

In [ ]:
# Compare model performance
for name, p in [('Linear Regression', p1), ('Random Forest', p2)]:

    print("\n", name)
    print("R2 Score:", r2_score(y_test, p))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, p)))
    print("MAE:", mean_absolute_error(y_test, p))

In [ ]:
# Predict fare for one test ride
fare = rf.predict(X_test.iloc[[0]])

print("Predicted Uber Fare: $", round(fare[0], 2))